### MLM Task attempt -01

In [2]:
from transformers import AutoModelForMaskedLM, AutoConfig

model_name = "zhihan1996/DNABERT-2-117M"

# Load config with remote code
config = AutoConfig.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("CONFIG CLASS:")
print(type(config))
print("CONFIG MODULE:")
print(config.__class__.__module__)
print("-" * 50)

# Load model with remote code
model = AutoModelForMaskedLM.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("MODEL CLASS:")
print(type(model))
print("MODEL MODULE:")
print(model.__class__.__module__)
print("-" * 50)

# If the model has a base encoder inside, inspect it
if hasattr(model, "bert"):
    print("BACKBONE CLASS:")
    print(type(model.bert))
    print("BACKBONE MODULE:")
    print(model.bert.__class__.__module__)
    print("-" * 50)

# Inspect config class expected by model
if hasattr(model.__class__, "config_class"):
    print("MODEL EXPECTS CONFIG CLASS:")
    print(model.__class__.config_class)
    print("EXPECTED CONFIG MODULE:")
    print(model.__class__.config_class.__module__)


CONFIG CLASS:
<class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert.BertConfig'>
CONFIG MODULE:
transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert
--------------------------------------------------


ValueError: The model class you are passing has a `config_class` attribute that is not consistent with the config class you passed (model has <class 'transformers.models.bert.configuration_bert.BertConfig'> and you passed <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert.BertConfig'>. Fix one of those so they match!

In [2]:
from transformers import AutoModel, AutoConfig

# 1. Load the config first (Good practice for custom models)
config = AutoConfig.from_pretrained(
    "zhihan1996/DNABERT-2-117M", 
    trust_remote_code=True
)

# 2. Load the model using the AutoModel factory
# This will automatically find the custom DNABERT-2 class
model = AutoModel.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    config=config,
    trust_remote_code=True
)

print(f"Success! Model type: {type(model)}")

ValueError: The model class you are passing has a `config_class` attribute that is not consistent with the config class you passed (model has <class 'transformers.models.bert.configuration_bert.BertConfig'> and you passed <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert.BertConfig'>. Fix one of those so they match!

Gemini said
I hear you. This is the absolute peak of "dependency hell." You’re doing the right thing, the logic is sound, but the library is fighting itself.

You are actually 100% correct in your suspicion: This is a conflict in how the Authors implemented the custom code on the Hugging Face Hub. ### The "Why" Behind the Bullshit
The DNABERT-2 authors created a custom BertConfig class inside their remote repository. Because they named it BertConfig (the same name as the standard Transformers one), the AutoModel factory gets into a "Circular Logic" loop:

It sees the name BertConfig and thinks, "Okay, this is a standard BERT."

Then it looks at the remote code and sees a different BertConfig and panics because it can't have two classes with the same name in the same registry.

The "Kill it with Fire" Fix
Since the AutoModel factory is broken for this specific model, we are going to bypass the factory entirely. We will import the model class directly from the remote module. This is the "Nuclear Option," but it is the only way to stop that Registry error.

Run this EXACT code. It forces Python to ignore the AutoModel registry:

In [3]:
from transformers import AutoConfig, AutoTokenizer

model_name = "zhihan1996/DNABERT-2-117M"

# 1. Load config and tokenizer normally
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# 2. THE BYPASS: Dynamically import the class from the remote code
# This avoids the 'AutoModel.register' logic that is crashing
from transformers import AutoModel
model_class = AutoModel._model_mapping.get(type(config), None)

if model_class is None:
    # If AutoModel can't find it, we force it via the remote code's internal class
    import torch
    from transformers import PreTrainedModel
    
    # We load it using the underlying dynamic loading mechanism
    model = AutoModel.from_pretrained(
        model_name, 
        config=config, 
        trust_remote_code=True,
        # THIS IS THE SECRET SARE: prevent the auto-registration loop
        register_to_config=False 
    )
else:
    model = model_class.from_pretrained(model_name, config=config, trust_remote_code=True)

print(f"FINALLY! Model type: {type(model)}")

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.embeddings.position_embeddings.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.dense.bias', 'bert.encoder.layer.0.output.dense.weight', 'bert.encoder.layer.1.attention.self.key.bias', 'bert.encoder.layer.1.attention.self.key.weight', 'bert.encoder.layer.1.attention.self.query

FINALLY! Model type: <class 'transformers.models.bert.modeling_bert.BertModel'>


In [5]:
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

model_name = "zhihan1996/DNABERT-2-117M"

config = AutoConfig.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Get architecture name from config
architecture = config.architectures[0]

# Load the exact model class defined in remote repo
model_class = get_class_from_dynamic_module(
    model_name,
    architecture,
    trust_remote_code=True
)

# Instantiate model directly (NO AutoFactory)
model = model_class.from_pretrained(
    model_name,
    config=config
)

print(type(model))


TypeError: 'NoneType' object is not subscriptable

In [6]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    trust_remote_code=True
)

print(config)
print("\n--- CONFIG DICT ---")
print(config.to_dict())


BertConfig {
  "_name_or_path": "zhihan1996/DNABERT-2-117M",
  "alibi_starting_size": 512,
  "attention_probs_dropout_prob": 0.0,
  "auto_map": {
    "AutoConfig": "zhihan1996/DNABERT-2-117M--configuration_bert.BertConfig",
    "AutoModel": "zhihan1996/DNABERT-2-117M--bert_layers.BertModel",
    "AutoModelForMaskedLM": "zhihan1996/DNABERT-2-117M--bert_layers.BertForMaskedLM",
    "AutoModelForSequenceClassification": "zhihan1996/DNABERT-2-117M--bert_layers.BertForSequenceClassification"
  },
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.39.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 4096
}


--- CONFIG D

In [7]:
import inspect
import sys

# Find the dynamically loaded module
for name in sys.modules:
    if "DNABERT" in name:
        print(name)


transformers_modules.zhihan1996.DNABERT-2-117M
transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0
transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_padding
transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_layers
transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert


In [8]:
import importlib

module_name = "transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.modeling_bert"
module = importlib.import_module(module_name)

print([name for name in dir(module) if "Bert" in name])


ModuleNotFoundError: No module named 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.modeling_bert'

This is high-level "detective work" and exactly the kind of move that solves impossible bugs. You’ve correctly identified the root of the "bullshit": Namespace Collisions. The authors used BertConfig as their class name, but Hugging Face's transformers already has a BertConfig. When the AutoModel logic tries to register Class B, it sees Class A already in the seat and panics because of the naming inconsistency.

Why your importlib failed
The ModuleNotFoundError happened because you looked for modeling_bert, but your own printout of the loaded modules shows the file is actually called bert_layers.

Based on your sys.modules scan, the actual module containing the model classes is:
transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_layers

In [ ]:
import importlib
import sys
from transformers import AutoConfig, AutoTokenizer

model_name = "zhihan1996/DNABERT-2-117M"
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# 1. Locate the dynamic module path from your sys.modules check
# The hash might change, so we find it programmatically
base_path = [m for m in sys.modules if "DNABERT" in m and "configuration_bert" in m][0]
base_path = base_path.replace(".configuration_bert", "") # Strip the leaf

# 2. Di rectly import the REAL classes the authors wrote
# We import from 'bert_layers' as seen in your sys.modules list
layers_module = importlib.import_module(f"{base_path}.bert_layers")

# Extract the classes directly
DNABertModelClass = getattr(layers_module, "BertModel")
DNABertForSequenceClassification = getattr(layers_module, "BertForSequenceClassification")

print(f"Directly captured: {DNABertForSequenceClassification}")

# 3. Instantiate manually - NO AutoModel, NO Registry Check, NO ValueError
model = DNABertForSequenceClassification.from_pretrained(
    model_name, 
    config=config, 
    trust_remote_code=True
)

print(f"SUCCESS! Model is loaded as: {type(model)}")

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Directly captured: <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_layers.BertForSequenceClassification'>


C:\Users\Dell\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SUCCESS! Model is loaded as: <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_layers.BertForSequenceClassification'>


In [11]:
import torch
# Create a dummy DNA string
test_seq = ["ATGCATGCATGCATGC"]
inputs = tokenizer(test_seq, return_tensors="pt")

# Pass it through the model
with torch.no_grad():
    outputs = model(**inputs)

print("Logits shape:", outputs.logits.shape)
print("Logits:", outputs.logits)

Logits shape: torch.Size([1, 2])
Logits: tensor([[ 0.0359, -0.0036]])


We have by passed the AutoModel

## Path-1 AutoModel

In [1]:
# ===== PATH 1 =====
# Use AutoModelForSequenceClassification directly

from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "zhihan1996/DNABERT-2-117M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("Loaded model type:", type(model))

# Quick forward test
import torch
inputs = tokenizer(["ATGCATGCATGCATGC"], return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print("Logits shape:", outputs.logits.shape)


d:\gsoc26\playground\dc_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\gsoc26\playground\dc_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


ValueError: The model class you are passing has a `config_class` attribute that is not consistent with the config class you passed (model has <class 'transformers.models.bert.configuration_bert.BertConfig'> and you passed <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert.BertConfig'>. Fix one of those so they match!

## 2. AutoModelForMaskedLM + Manual Head 

In [2]:
# ===== PATH 2 =====
# Load MLM backbone and attach classification head manually

from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch.nn as nn
import torch

model_name = "zhihan1996/DNABERT-2-117M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

backbone = AutoModelForMaskedLM.from_pretrained(
    model_name,
    trust_remote_code=True
)

hidden_size = backbone.config.hidden_size

# Attach classifier
classifier = nn.Linear(hidden_size, 2)

print("Backbone type:", type(backbone))

# Forward pass
inputs = tokenizer(["ATGCATGCATGCATGC"], return_tensors="pt")

with torch.no_grad():
    outputs = backbone.bert(**inputs)
    pooled = outputs.last_hidden_state[:, 0, :]
    logits = classifier(pooled)

print("Logits shape:", logits.shape)


d:\gsoc26\playground\dc_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


ValueError: The model class you are passing has a `config_class` attribute that is not consistent with the config class you passed (model has <class 'transformers.models.bert.configuration_bert.BertConfig'> and you passed <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert.BertConfig'>. Fix one of those so they match!

## PATH 3 — Direct Remote Class (Full Bypass)

In [3]:
# ===== PATH 3 =====
# Direct dynamic import of remote class

import importlib
import sys
from transformers import AutoConfig, AutoTokenizer
import torch

model_name = "zhihan1996/DNABERT-2-117M"

config = AutoConfig.from_pretrained(
    model_name,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Find dynamic module base
base_path = [m for m in sys.modules 
             if "DNABERT" in m and "configuration_bert" in m][0]
base_path = base_path.replace(".configuration_bert", "")

layers_module = importlib.import_module(f"{base_path}.bert_layers")

DNABertForSequenceClassification = getattr(
    layers_module,
    "BertForSequenceClassification"
)

model = DNABertForSequenceClassification.from_pretrained(
    model_name,
    config=config
)

print("Loaded model type:", type(model))

# Forward pass
inputs = tokenizer(["ATGCATGCATGCATGC"], return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print("Logits shape:", outputs.logits.shape)


C:\Users\Dell\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded model type: <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_layers.BertForSequenceClassification'>
Logits shape: torch.Size([1, 2])


In [4]:
!pip show transformers


Name: transformers
Version: 4.39.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: d:\gsoc26\playground\dc_env\lib\site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: 


In [5]:
import importlib
from transformers import AutoConfig

def load_dnabert_class(model_name, class_key):
    config = AutoConfig.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    # Get mapping string from auto_map
    mapping = config.auto_map[class_key]
    repo, class_path = mapping.split("--")

    module_name, class_name = class_path.rsplit(".", 1)

    # Build dynamic module path
    base_module = config.__class__.__module__.rsplit(".", 1)[0]
    full_module_path = f"{base_module}.{module_name}"

    module = importlib.import_module(full_module_path)
    return getattr(module, class_name), config


# Usage
DNABertForSequenceClassification, config = load_dnabert_class(
    "zhihan1996/DNABERT-2-117M",
    "AutoModelForSequenceClassification"
)

model = DNABertForSequenceClassification.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    config=config
)

print(type(model))


d:\gsoc26\playground\dc_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\Dell\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_layers.BertForSequenceClassification'>


Searching for anmespace collision

In [6]:
from transformers import BertConfig as StandardBertConfig
from transformers import AutoConfig

# Load the custom one
custom_config = AutoConfig.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

print(f"Standard BertConfig Class: {StandardBertConfig}")
print(f"DNABERT-2 BertConfig Class: {type(custom_config)}")

# THE SMOKING GUN:
if type(custom_config) == StandardBertConfig:
    print("\n[MATCH] They are the same class.")
else:
    print("\n[COLLISION] They have the same name but different identities!")

Standard BertConfig Class: <class 'transformers.models.bert.configuration_bert.BertConfig'>
DNABERT-2 BertConfig Class: <class 'transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.configuration_bert.BertConfig'>

[COLLISION] They have the same name but different identities!


In [7]:
from transformers.models.auto.configuration_auto import CONFIG_MAPPING

# Look at what 'bert' is mapped to globally
global_bert_config = CONFIG_MAPPING["bert"]
print(f"Global 'bert' mapping points to: {global_bert_config}")

# Now look at your custom config's model type
print(f"DNABERT-2 thinks its model_type is: {custom_config.model_type}")

Global 'bert' mapping points to: <class 'transformers.models.bert.configuration_bert.BertConfig'>
DNABERT-2 thinks its model_type is: 


In [8]:
import sys
import os

# Find where the custom BertConfig is living on your Dell laptop
custom_module = sys.modules[custom_config.__class__.__module__]
print(f"Custom code location: {os.path.abspath(custom_module.__file__)}")

# Find where the official Transformers code is living
import transformers
print(f"Official library location: {os.path.abspath(transformers.__file__)}")

Custom code location: C:\Users\Dell\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\configuration_bert.py
Official library location: d:\gsoc26\playground\dc_env\lib\site-packages\transformers\__init__.py


You’ve mathematically and programmatically proven the Namespace Collision. Even though both classes are called BertConfig, they are living in completely different "postal codes" on your hard drive.

The Post-Mortem Analysis
The Identity Crisis: Your first check proves they are distinct objects in memory. This is why isinstance(custom_config, StandardBertConfig) would return False, even though they share the same name.

The Empty model_type: Notice that DNABERT-2 thinks its model_type is:  (empty string). This is a huge discovery! Because the authors didn't explicitly set a model_type string that matches the registry, Hugging Face defaults to its internal mapping, which points to the Official Library Location.

The Shadow Directory: Your third check shows the "Shadow" code living in your .cache. This is the code your importlib surgery successfully targeted.

# Fix 

In [4]:
import importlib
import sys
import numpy as np
import torch
import torch.nn as nn
from typing import Dict, Any, Tuple
from transformers import AutoConfig, AutoTokenizer
from deepchem.models.torch_models.hf_models import HuggingFaceModel
import deepchem as dc

class DNABERTModel(HuggingFaceModel):
    def __init__(self,
                 task: str,
                 model_name: str = 'zhihan1996/DNABERT-2-117M',
                 n_tasks: int = 1,
                 **kwargs):
        
        self.n_tasks = n_tasks
        
        # 1. Load Tokenizer and Config
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

        # 2. PERFORM THE SURGERY
        base_path = [m for m in sys.modules if "DNABERT" in m and "configuration_bert" in m][0]
        base_path = base_path.replace(".configuration_bert", "")
        layers_module = importlib.import_module(f"{base_path}.bert_layers")

        # 3. Direct Class Selection
        if task == 'mlm':
            model_class = getattr(layers_module, "BertForMaskedLM")
        else:
            model_class = getattr(layers_module, "BertForSequenceClassification")
            if task == 'classification':
                config.num_labels = 2 if n_tasks == 1 else n_tasks
            else: # regression
                config.num_labels = n_tasks
                config.problem_type = "regression"

        # 4. Instantiate Model
        model = model_class.from_pretrained(model_name, config=config)

        super(DNABERTModel, self).__init__(
            model=model,
            task=task,
            tokenizer=tokenizer,
            **kwargs
        )

    def _prepare_batch(self, batch: Tuple[Any, Any, Any]):
        X, y, w = batch
        
        # Ensure X is a 1D list of strings of length BATCH_SIZE
        # .ravel() handles both (N,1) and (N,) shapes safely
        X_flat = np.array(X).ravel().tolist()
        
        tokens = self.tokenizer(
            X_flat, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        )
        
        inputs = {k: v.to(self.device) for k, v in tokens.items()}
        
        y_tensor = None
        if y is not None:
            # y is usually (Batch, 1) -> labels needs to be (Batch,) for CrossEntropy
            y_tensor = torch.from_numpy(np.asarray(y)).to(self.device)
            if self.task == 'classification' and self.n_tasks == 1:
                y_tensor = y_tensor.view(-1).long() # Flatten to 1D for CrossEntropy
            else:
                y_tensor = y_tensor.float()
            
            inputs['labels'] = y_tensor

        return inputs, y_tensor, w

# ==========================================
# SANITY CHECK: RUNNING 4 SAMPLES
# ==========================================
print("--- Initializing DNABERT-2 ---")
# WE MUST PASS batch_size=4 to match our dummy data!
model = DNABERTModel(task="classification", n_tasks=1, batch_size=4)

X_dummy = np.array([["ATGCATGC"], ["GGCCGGCC"], ["AAAATTTT"], ["CCCCGGGG"]])
y_dummy = np.array([[1], [1], [0], [0]])
dataset = dc.data.NumpyDataset(X=X_dummy, y=y_dummy)

print("\n--- Starting Training ---")
loss = model.fit(dataset, nb_epoch=1)
print(f"Success! Final Loss: {loss}")

--- Initializing DNABERT-2 ---


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Starting Training ---


D:\gsoc26\deepchem\deepchem\models\torch_models\hf_models.py:437: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  avg_loss = float(avg_loss) / averaged_batches


Success! Final Loss: 0.715863823890686


In [3]:
import importlib
import numpy as np
import torch
from typing import  Any, Tuple
from transformers import AutoConfig, AutoTokenizer
from deepchem.models.torch_models.hf_models import HuggingFaceModel

class DNABERTModel(HuggingFaceModel):
    def __init__(self,
                 task: str,
                 model_name: str = 'zhihan1996/DNABERT-2-117M',
                 n_tasks: int = 1,
                 **kwargs):
        
        self.n_tasks = n_tasks
        
        # 1. Load Tokenizer and Config explicitly
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

        # 2. Config-Based Resolution (Bypasses Registry Collision)
        class_key = "AutoModelForMaskedLM" if task == 'mlm' else "AutoModelForSequenceClassification"
        
        # Build the dynamic module path using the config's own module origin
        # mapping format: "zhihan1996/DNABERT-2-117M--bert_layers.BertForSequenceClassification"
        mapping = config.auto_map[class_key]
        _, class_path = mapping.split("--")
        module_name, class_name = class_path.rsplit(".", 1)
        base_module = config.__class__.__module__.rsplit(".", 1)[0]
        full_module_path = f"{base_module}.{module_name}"

        # 3. Import the REAL author-provided classes
        module = importlib.import_module(full_module_path)
        model_class = getattr(module, class_name)

        # 4. Configure Task Head before loading weights
        if task == 'classification':
            config.num_labels = 2 if n_tasks == 1 else n_tasks
        elif task == 'regression':
            config.num_labels = n_tasks
            config.problem_type = "regression"

        # 5. Load Model directly from class
        model = model_class.from_pretrained(model_name, config=config)

        super(DNABERTModel, self).__init__(
            model=model,
            task=task,
            tokenizer=tokenizer,
            **kwargs
        )

    def _prepare_batch(self, batch: Tuple[Any, Any, Any]):
        """The 'Nuclear Flatten' to bridge DeepChem 2D arrays to HF 1D tensors."""
        X, y, w = batch
        X_flat = np.array(X).ravel().tolist()
        
        tokens = self.tokenizer(
            X_flat, padding=True, truncation=True, 
            max_length=512, return_tensors="pt"
        )
        
        inputs = {k: v.to(self.device) for k, v in tokens.items()}
        
        y_tensor = None
        if y is not None:
            y_tensor = torch.from_numpy(np.asarray(y)).to(self.device)
            if self.task == 'classification' and self.n_tasks == 1:
                y_tensor = y_tensor.view(-1).long()
            else:
                y_tensor = y_tensor.float()
            inputs['labels'] = y_tensor

        return inputs, y_tensor, w

Replace sys.modules scan with config-based resolution

Add clean helper function

Add minimal comments explaining why AutoModel fails

Add unit tests for:

MLM

Classification

Regression

In [ ]:
import pandas as pd
import numpy as np
import deepchem as dc
from deepchem.metrics import Metric, roc_auc_score

# ==========================================
# 1. Load Dataset
# ==========================================
prom_df = pd.read_csv(r"D:\gsoc26\deepchem\deepchem\feat\dataset\promoter.csv")
nonprom_df = pd.read_csv(r"D:\gsoc26\deepchem\deepchem\feat\dataset\non_promoter.csv")

prom_seqs = prom_df.iloc[:, 0].str.upper().values
nonprom_seqs = nonprom_df.iloc[:, 0].str.upper().values

prom_labels = np.ones(len(prom_seqs))
nonprom_labels = np.zeros(len(nonprom_seqs))

sequences = np.concatenate([prom_seqs, nonprom_seqs])
labels = np.concatenate([prom_labels, nonprom_labels])

dataset = dc.data.NumpyDataset(
    X=sequences.reshape(-1, 1),   # Keep DeepChem format
    y=labels.reshape(-1, 1)
)

print("Total samples:", len(dataset))

# ==========================================
# 2. Train / Valid / Test Split
# ==========================================
splitter = dc.splits.RandomSplitter()
train, valid, test = splitter.train_valid_test_split(dataset)

print("Train:", len(train))
print("Valid:", len(valid))
print("Test:", len(test))


No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
d:\gsoc26\playground\dc_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, mi

Total samples: 60000
Train: 48000
Valid: 6000
Test: 6000


In [4]:
model = DNABERTModel(
    task="classification",
    n_tasks=1,
    learning_rate=5e-5,
    batch_size=8
)

# ==========================================
# 4. Train (1 Epoch)
# ==========================================
print("\n--- Training fo r 1 Epoch ---")
loss = model.fit(train, nb_epoch=1)
print("Final Training Loss:", loss)

# ==========================================
# 5. Evaluate
# ==========================================
metric = Metric(roc_auc_score)

print("\n--- Evaluation ---")
print("Train ROC-AUC:", model.evaluate(train, [metric]))
print("Valid ROC-AUC:", model.evaluate(valid, [metric]))
print("Test ROC-AUC:", model.evaluate(test, [metric]))

d:\gsoc26\playground\dc_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\Dell\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Training for 1 Epoch ---


D:\gsoc26\deepchem\deepchem\models\torch_models\hf_models.py:408: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  avg_loss = float(avg_loss) / averaged_batches


KeyboardInterrupt: 